# STT evaluation: WER & TER

Scores each STT provider's transcript against the hand-corrected **golden** reference.

- **WER** (Word Error Rate) = `(substitutions + deletions + insertions) / reference_words`. Lower is better; `0` is perfect.
- **TER** (Translation Edit Rate) = like WER but also credits *shifts* (a phrase moved as a block counts as one edit, not delete+insert). Reported as a percentage.

Inputs live under `data/golden-label/<lecture>/`:

| file | role |
|---|---|
| `golden.md`  | reference (ground truth) |
| `whisper.md` | hypothesis — `whisper_cpp` output |
| `google.md`  | hypothesis — `google_stt` output |

All files share the line format `**[H:MM:SS – H:MM:SS]** text`. We strip the timestamp markers, concatenate the text, then normalize (lowercase, drop punctuation) before scoring — the usual convention so casing/punctuation don't dominate the error rate.

## Setup

`jiwer`, `sacrebleu`, and `pandas` aren't in the project's default deps. Install them into the active env if missing (the project's private index needs auth, so pull these from public PyPI):

```bash
python -m pip install -i https://pypi.org/simple jiwer sacrebleu pandas
```

In [1]:
import re
from pathlib import Path

import jiwer
import pandas as pd
from sacrebleu.metrics import TER

In [2]:
# --- config ---------------------------------------------------------------
# Resolve the repo root whether the notebook runs from repo root or notebooks/.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "golden-label").exists())
GOLDEN_ROOT = REPO_ROOT / "data" / "golden-label"

REFERENCE_FILE = "golden.md"
# hypothesis filename -> provider label shown in the results table
PROVIDERS = {"whisper.md": "whisper_cpp", "google.md": "google_stt"}

print("repo root:", REPO_ROOT)
print("golden labels:", GOLDEN_ROOT)

repo root: /Users/shamilarslanov/CurrentProjects/autosummary
golden labels: /Users/shamilarslanov/CurrentProjects/autosummary/data/golden-label


## Parsing & normalization

In [3]:
# Matches lines like: **[2:01:41 – 2:01:47]** Item number three for today.
_LINE = re.compile(r"^\s*\*\*\[[^\]]*\]\*\*\s*(.*)$")


def load_transcript(path: Path) -> str:
    """Concatenate the spoken text from a timestamped `.md` transcript into one string."""
    parts = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        m = _LINE.match(line)
        if m and m.group(1).strip():
            parts.append(m.group(1).strip())
    return " ".join(parts)


# Standard WER pre-processing: case- and punctuation-insensitive.
_NORMALIZE = jiwer.Compose(
    [
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.ReduceToListOfListOfWords(),
    ]
)


def normalized_str(text: str) -> str:
    """Apply the same normalization as WER, flattened back to a string (for TER)."""
    return " ".join(w for sentence in _NORMALIZE(text) for w in sentence)

## Score every golden-labelled lecture × provider

In [4]:
ter_metric = TER()
rows = []

for lecture_dir in sorted(p for p in GOLDEN_ROOT.iterdir() if p.is_dir()):
    ref_path = lecture_dir / REFERENCE_FILE
    if not ref_path.exists():
        continue
    reference = load_transcript(ref_path)
    ref_norm = normalized_str(reference)

    for hyp_file, provider in PROVIDERS.items():
        hyp_path = lecture_dir / hyp_file
        if not hyp_path.exists():
            continue
        hypothesis = load_transcript(hyp_path)

        out = jiwer.process_words(
            reference,
            hypothesis,
            reference_transform=_NORMALIZE,
            hypothesis_transform=_NORMALIZE,
        )
        ter = ter_metric.corpus_score([normalized_str(hypothesis)], [[ref_norm]]).score

        rows.append(
            {
                "lecture": lecture_dir.name,
                "provider": provider,
                "ref_words": sum(len(s) for s in _NORMALIZE(reference)),
                "WER": out.wer,
                "TER": ter / 100.0,
                "sub": out.substitutions,
                "del": out.deletions,
                "ins": out.insertions,
                "hits": out.hits,
            }
        )

results = pd.DataFrame(rows)
results

,lecture,provider,ref_words,WER,TER,sub,del,ins,hits
0,lecture-06,whisper_cpp,2090,0.022967,0.022967,7,41,0,2042
1,lecture-06,google_stt,2090,0.162679,0.161722,193,38,109,1859


In [5]:
# Pretty view: rates as percentages, sorted best-first per lecture.
view = results.copy()
view["WER"] = (view["WER"] * 100).round(2).astype(str) + " %"
view["TER"] = (view["TER"] * 100).round(2).astype(str) + " %"
view.sort_values(["lecture", "provider"])

,lecture,provider,ref_words,WER,TER,sub,del,ins,hits
1,lecture-06,google_stt,2090,16.27 %,16.17 %,193,38,109,1859
0,lecture-06,whisper_cpp,2090,2.3 %,2.3 %,7,41,0,2042


## Inspect the errors

A word-level alignment for one lecture/provider — handy for sanity-checking *what* the STT got wrong (`S`=substitution, `D`=deletion, `I`=insertion).

In [6]:
LECTURE = "lecture-06"
HYP_FILE = "google.md"  # switch to "whisper.md" to inspect the other provider

lecture_dir = GOLDEN_ROOT / LECTURE
alignment = jiwer.process_words(
    load_transcript(lecture_dir / REFERENCE_FILE),
    load_transcript(lecture_dir / HYP_FILE),
    reference_transform=_NORMALIZE,
    hypothesis_transform=_NORMALIZE,
)
print(jiwer.visualize_alignment(alignment, skip_correct=True))

=== SENTENCE 1 ===

REF: item number three for today ** selfattention well to actually discuss **** selfattention i need to take a step back and i need to talk to you surprisingly about information retrieval which is not the subject of our class but turns out to be very relevant so how does search work in deep learning how does information retrieval work when you have neural networks well basically it works like this you have the queries query one query two and you have documents document one document two document three document four and in usual information retrieval what you would do you would establish a similarity you need to establish some kind of similarity between queries and documents this kind of similarity will be called relevance and you will train it to be the relevance ***** so it should be higher when this document is relevant to this query this is how search generally works **** whatever it is but when you have deep learning when you have these neural networks that can p